# Loading Libraries

In [1]:
from cso_classifier import CSOClassifier
import json

# Loading Paper

In [2]:
paper = {
        "title": "De-anonymizing Social Networks",
        "abstract": "Operators of online social networks are increasingly sharing potentially "
        "sensitive information about users and their relationships with advertisers, application "
        "developers, and data-mining researchers. Privacy is typically protected by anonymization, "
        "i.e., removing names, addresses, etc. We present a framework for analyzing privacy and "
        "anonymity in social networks and develop a new re-identification algorithm targeting "
        "anonymized social-network graphs. To demonstrate its effectiveness on real-world networks, "
        "we show that a third of the users who can be verified to have accounts on both Twitter, a "
        "popular microblogging service, and Flickr, an online photo-sharing site, can be re-identified "
        "in the anonymous Twitter graph with only a 12% error rate. Our de-anonymization algorithm is "
        "based purely on the network topology, does not require creation of a large number of dummy "
        "\"sybil\" nodes, is robust to noise and all existing defenses, and works even when the overlap "
        "between the target network and the adversary's auxiliary information is small.",
        "keywords": "data mining, data privacy, graph theory, social networking (online)"
        }

In [3]:
"""
paper = {
        "title": "Generative Adversarial Networks for Synthetic Data Generation in Finance: Evaluating Statistical Similarities and Quality Assessment",
        "abstract": "Generating synthetic data is a complex task that necessitates accurately replicating the statistical and mathematical properties "
        "of the original data elements. In sectors such as finance, utilizing and disseminating real data for research or model development "
        "can pose substantial privacy risks owing to the inclusion of sensitive information. Additionally, authentic data may be scarce, "
        "particularly in specialized domains where acquiring ample, varied, and high-quality data is difficult or costly. This scarcity or "
        "limited data availability can limit the training and testing of machine-learning models. In this paper, we address this challenge. "
        "In particular, our task is to synthesize a dataset with similar properties to an input dataset about the stock market. The input "
        "dataset is anonymized and consists of very few columns and rows, contains many inconsistencies, such as missing rows and duplicates, "
        "and its values are not normalized, scaled, or balanced. We explore the utilization of generative adversarial networks, a deep-learning "
        "technique, to generate synthetic data and evaluate its quality compared to the input stock dataset. Our innovation involves generating "
        "artificial datasets that mimic the statistical properties of the input elements without revealing complete information. For example, "
        "synthetic datasets can capture the distribution of stock prices, trading volumes, and market trends observed in the original dataset. "
        "The generated datasets cover a wider range of scenarios and variations, enabling researchers and practitioners to explore "
        "different market conditions and investment strategies. This diversity can enhance the robustness and generalization of "
        "machine-learning models. We evaluate our synthetic data in terms of the mean, similarities, and correlations.",
        "keywords": "generative adversarial networks; deep learning; data augmentation; synthetic data"
        }

"""

'\npaper = {\n        "title": "Generative Adversarial Networks for Synthetic Data Generation in Finance: Evaluating Statistical Similarities and Quality Assessment",\n        "abstract": "Generating synthetic data is a complex task that necessitates accurately replicating the statistical and mathematical properties "\n        "of the original data elements. In sectors such as finance, utilizing and disseminating real data for research or model development "\n        "can pose substantial privacy risks owing to the inclusion of sensitive information. Additionally, authentic data may be scarce, "\n        "particularly in specialized domains where acquiring ample, varied, and high-quality data is difficult or costly. This scarcity or "\n        "limited data availability can limit the training and testing of machine-learning models. In this paper, we address this challenge. "\n        "In particular, our task is to synthesize a dataset with similar properties to an input dataset about t

In [4]:
from IPython.display import display, HTML

display(HTML('<h1>'+paper["title"]+'</h1>'))
display(HTML('<p><strong>Abstract:</strong> '+paper["abstract"]+'</p>'))
display(HTML('<p><strong>Keywords:</strong> <i>'+paper["keywords"]+'</i></p>'))

# Run Classifier

In [5]:
#cc = CSOClassifier(explanation=True, get_weights=True, filter_by=["computer security"])
cc = CSOClassifier(explanation=True, get_weights=True)


result = cc.run(paper)

Computer Science Ontology loaded.
Cached model loaded.
Word2Vec model loaded (legacy pickle).


# Printing and Saving

In [6]:
print(result)

with open('output.json', 'w') as outfile:
    json.dump(result, outfile, indent=4)

{'syntactic': ['online social networks', 'anonymization', 'real-world networks', 'data mining', 'social networks', 'twitter', 'anonymity', 'network topology', 'sensitive informations', 'privacy', 'data privacy', 'graph theory', 'micro-blog'], 'semantic': ['online social networks', 'online communities', 'social networking sites', 'anonymity', 'network topology', 'data privacy', 'graph theory', 'topology', 'anonymous communication', 'network architecture', 'social media', 'privacy', 'association rules', 'micro-blog', 'social networks', 'twitter', 'anonymization', 'bipartite graphs', 'data mining'], 'union': ['online social networks', 'online communities', 'real-world networks', 'social networks', 'twitter', 'social networking sites', 'anonymity', 'network topology', 'data privacy', 'graph theory', 'topology', 'anonymous communication', 'anonymization', 'bipartite graphs', 'network architecture', 'data mining', 'social media', 'sensitive informations', 'privacy', 'association rules', 'mic

## Evaluation of CSO Classifier 4.0 on Paper dataset

In [ ]:
from cso_classifier import CSOClassifier
import json
import pandas as pd
from pathlib import Path

# ---- config ----
CSV_PATH = Path(r"C:\Users\Faisal Ramzan\Desktop\KMI WORK CSO 4.0\cso_classifier_upgradation\cso-classifier-master\evaluation_dataset\paper_dataset.csv")
LIMIT = 10000                 # max rows to consider
CHECKPOINT_SIZE = 100         # save/flush every N papers
OUT_JSONL = Path("output.jsonl")  # checkpoint file (appendable)
FINAL_JSON = Path("output.json")  # final merged array when complete
# -----------------

# Load dataset
df = pd.read_csv(CSV_PATH)
if LIMIT is not None:
    df = df.head(LIMIT)

# Initialize classifier (keep your current setting)
cc = CSOClassifier(explanation=True, get_weights=True, filter_by=["computer security"])

# Resume support: collect already processed indices from JSONL (if exists)
seen = set()
if OUT_JSONL.exists():
    with OUT_JSONL.open("r", encoding="utf-8") as fh:
        for line in fh:
            try:
                obj = json.loads(line)
                if "index" in obj:
                    seen.add(obj["index"])
            except Exception:
                continue

print(f"[info] Loaded {len(df)} rows. Already processed: {len(seen)}")

# Open JSONL in append mode
processed_this_run = 0
buffer_since_flush = 0
OUT_JSONL.parent.mkdir(parents=True, exist_ok=True)
fh = OUT_JSONL.open("a", encoding="utf-8")

try:
    for idx, row in df.iterrows():
        if idx in seen:
            continue

        paper = {
            "title": str(row.get("title", "")),
            "abstract": str(row.get("abstract", "")),
            "keywords": str(row.get("keywords", "")) if "keywords" in df.columns else ""
        }

        try:
            result = cc.run(paper)
            rec = {
                "index": idx,
                "title": paper["title"],
                "result": result
            }
        except Exception as e:
            # record the error but keep the checkpoint trail
            rec = {
                "index": idx,
                "title": paper.get("title", ""),
                "error": str(e)
            }

        # Append one JSON object per line (safe to resume)
        fh.write(json.dumps(rec, ensure_ascii=False) + "\n")
        buffer_since_flush += 1
        processed_this_run += 1

        # Flush at each checkpoint
        if buffer_since_flush >= CHECKPOINT_SIZE:
            fh.flush()
            try:
                import os
                os.fsync(fh.fileno())
            except Exception:
                pass
            print(f"[checkpoint] processed {processed_this_run} in this run; last index = {idx}")
            buffer_since_flush = 0

finally:
    # Final flush
    fh.flush()
    try:
        import os
        os.fsync(fh.fileno())
    except Exception:
        pass
    fh.close()

print(f"[info] This run processed {processed_this_run} new rows.")

# Merge JSONL -> single JSON array (optional finalization step)
# If the run was interrupted, you can skip this step and resume later;
# once fully done, re-run from here to produce the consolidated JSON.
all_results = []
with OUT_JSONL.open("r", encoding="utf-8") as fh2:
    for line in fh2:
        try:
            all_results.append(json.loads(line))
        except Exception:
            continue

with FINAL_JSON.open("w", encoding="utf-8") as out_f:
    json.dump(all_results, out_f, indent=4, ensure_ascii=False)

print(f"[done] Saved consolidated results to {FINAL_JSON} (total records: {len(all_results)})")


[info] Loaded 10000 rows. Already processed: 0
Computer Science Ontology loaded.
Cached model loaded.
Word2Vec model loaded (legacy pickle).
[checkpoint] processed 100 in this run; last index = 99
[checkpoint] processed 200 in this run; last index = 199
[checkpoint] processed 300 in this run; last index = 299
[checkpoint] processed 400 in this run; last index = 399
[checkpoint] processed 500 in this run; last index = 499
[checkpoint] processed 600 in this run; last index = 599
[checkpoint] processed 700 in this run; last index = 699
[checkpoint] processed 800 in this run; last index = 799
[checkpoint] processed 900 in this run; last index = 899
[checkpoint] processed 1000 in this run; last index = 999
[checkpoint] processed 1100 in this run; last index = 1099
[checkpoint] processed 1200 in this run; last index = 1199
[checkpoint] processed 1300 in this run; last index = 1299
[checkpoint] processed 1400 in this run; last index = 1399
[checkpoint] processed 1500 in this run; last index =

## Check W2V for hyphen - topics and filter them

In [13]:
import pickle

# Path to your pickled KeyedVectors model
#model_path = r"C:\Users\Faisal Ramzan\Desktop\cso_classifier_upgradation\cso-classifier-master\cso_classifier\assets\model.p"
model_path = r"C:\Users\Faisal Ramzan\Desktop\KMI WORK CSO 4.0\cso_classifier_upgradation\cso-classifier-master\cso_classifier\assets\model.p"

# Load the KeyedVectors model
with open(model_path, "rb") as f:
    model = pickle.load(f)

# Extract vocabulary
vocab = list(model.key_to_index.keys())

# Total words
total_words = len(vocab)

# Filter words containing hyphen
hyphen_words = [word for word in vocab if "-" in word]
total_hyphen_words = len(hyphen_words)

# Percentage
percentage_hyphen = (total_hyphen_words / total_words) * 100

# Sort by frequency (descending)
hyphen_words_sorted = sorted(
    hyphen_words, key=lambda w: model.get_vecattr(w, "count"), reverse=True
)

# Print stats
print(f"Total words in model: {total_words}")
print(f"Words with hyphen: {total_hyphen_words}")
print(f"Percentage of hyphen words: {percentage_hyphen:.2f}%\n")

# Print top 50
print("Top 50 words with hyphens:\n")
for i, word in enumerate(hyphen_words_sorted[:200], start=1):
    count = model.get_vecattr(word, "count")
    print(f"{i}. {word} ({count})")


Total words in model: 2074429
Words with hyphen: 410256
Percentage of hyphen words: 19.78%

Top 50 words with hyphens:

1. - (2074284)
2. long-term (2073653)
3. covid-19 (2073348)
4. follow-up (2073261)
5. real-time (2072854)
6. large-scale (2072340)
7. three-dimensional (2072213)
8. x-ray (2072183)
9. decision-making (2072050)
10. -1 (2072043)
11. meta-analysis (2071907)
12. covid-19_pandemic (2071832)
13. cross-sectional (2071811)
14. short-term (2071638)
15. two-dimensional (2071622)
16. well-being (2071388)
17. state-of-the-art (2071256)
18. so-called (2071077)
19. well-known (2071013)
20. real-world (2070850)
21. -0 (2070828)
22. high-risk (2070498)
23. high-quality (2070441)
24. -- (2070335)
25. self-efficacy (2070251)
26. high-resolution (2070242)
27. sars-cov-2 (2070058)
28. non-linear (2070050)
29. cost-effective (2069952)
30. in-depth (2069782)
31. socio-economic (2069751)
32. low-cost (2069568)
33. anti-inflammatory (2069545)
34. il-6 (2069452)
35. high-speed (2069421)
36. o

In [14]:
import pickle

# Path to your pickled KeyedVectors model
model_path = r"C:\Users\Faisal Ramzan\Desktop\KMI WORK CSO 4.0\cso_classifier_upgradation\cso-classifier-master\cso_classifier\assets\model.p"

# Load the KeyedVectors model
with open(model_path, "rb") as f:
    model = pickle.load(f)

# Extract vocabulary
vocab = list(model.key_to_index.keys())

# Count total words
total_words = len(vocab)

# Filter words containing hyphen
hyphen_words = [word for word in vocab if "-" in word]
total_hyphen_words = len(hyphen_words)

# Percentage of hyphenated words
percentage_hyphen = (total_hyphen_words / total_words) * 100

print(f"Total words in model: {total_words}")
print(f"Words with hyphen: {total_hyphen_words}")
print(f"Percentage of hyphen words: {percentage_hyphen:.2f}%")

# --- Query Function ---
def query_word(word, topn=10):
    """Retrieve most similar words for a given query word"""
    if word in model.key_to_index:
        similar = model.most_similar(word, topn=topn)
        print(f"\nTop {topn} words similar to '{word}':\n")
        for i, (w, score) in enumerate(similar, start=1):
            print(f"{i}. {w} (similarity: {score:.4f})")
    else:
        print(f"\n'{word}' not found in vocabulary.")

# Example query
query_word("data-", topn=100)



Total words in model: 2074429
Words with hyphen: 410256
Percentage of hyphen words: 19.78%

Top 100 words similar to 'data-':

1. data-intensive (similarity: 0.5621)
2. data_management (similarity: 0.5526)
3. data-intensive_systems (similarity: 0.5375)
4. data-management (similarity: 0.5366)
5. database_systems (similarity: 0.5291)
6. macroprogramming (similarity: 0.5249)
7. network_tra_c (similarity: 0.5204)
8. database-centric (similarity: 0.5174)
9. streaming_data_processing (similarity: 0.5167)
10. data-intensive_scientific (similarity: 0.5155)
11. virtualized_servers (similarity: 0.5140)
12. olap_cubes (similarity: 0.5137)
13. hpc_infrastructures (similarity: 0.5132)
14. ar-chitectures (similarity: 0.5129)
15. big_data_streaming (similarity: 0.5129)
16. datacentric (similarity: 0.5127)
17. stream_processing_engine (similarity: 0.5097)
18. data_management_systems (similarity: 0.5096)
19. resource_brokering (similarity: 0.5084)
20. ontology-supported (similarity: 0.5083)
21. column-